# Chapter 16 — Restart Is Not Resume

**Companion to *Applied AI*.**

This notebook accompanies Chapter 16. Its evidence is unusually good: producers
were **really killed** with no cleanup, at durable checkpoints, and provider
effects were counted **outside the ledger** by a synthetic provider that
appended a line to its own receipt log for every request it received.

So the ledger's claims can be checked against something it did not write.

## Question

**Could the effect have happened?**

## What this notebook establishes

- Each preserved case classified from its ledger alone: stage, provider effect,
  next operation, and whether repeating risks a duplicate.
- The chapter's central pair, read from the receipt logs: resuming after the
  manifest produced **one** provider request; a naive restart of an identical
  copy produced **two**.
- A resume refused because the recorded intent no longer reproduces (drift).
- A resume refused because the effect is **unknown** — the honest answer, and
  the whole recovery plan.

## What this notebook does **not** establish

- **Not crash atomicity.** Every kill happened at a durable checkpoint.
- **Not exactly-once.** An unknown effect is refused, not resolved.
- **Not concurrency.** Two processes resuming the same call could both start it.
- The provider was **synthetic** and the fixture was one task.

## Setup

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="working-state"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
RUN = EVIDENCE_DIR / "working-state" / "2026-09-13-68f4ba0"
CASES = sorted(p.name for p in RUN.iterdir()
               if p.is_dir() and p.name not in {"tests", "failure-injection"})
print("run   :", RUN.name)
print("cases :")
for c in CASES:
    print("   -", c)

run   : 2026-09-13-68f4ba0
cases :
   - clean-complete
   - compilation-failed
   - killed-after-manifest
   - killed-after-manifest-drift
   - killed-after-observation
   - killed-during-compilation
   - killed-during-effect
   - killed-during-effect-naive-restart


## 1. Restart is not resume

**Restart** means the durable state survived the process. **Resume** means
another process can decide what to do next from what is recorded, without
asking the person who was watching.

It has to answer seven questions.

In [2]:
QUESTIONS = [
    "What was being attempted?",
    "What inputs were available?",
    "What was required?",
    "What failed?",
    "What succeeded?",
    "What remains unresolved?",
    "Which next operation is safe?",
]
for q in QUESTIONS:
    print("  -", q)
print()
print("A database that survives a crash answers none of these by itself.")
print("They are questions about the WORK, not about the storage.")

  - What was being attempted?
  - What inputs were available?
  - What was required?
  - What failed?
  - What succeeded?
  - What remains unresolved?
  - Which next operation is safe?

A database that survives a crash answers none of these by itself.
They are questions about the WORK, not about the storage.


## 2. What each interrupted case projects

Read the work-state projection each inspecting process recorded. Note the
`provider_receipts` column: that count comes from the provider's own log, not
from the ledger.

In [3]:
def load(case, name):
    p = RUN / case / name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

rows = []
for case in CASES:
    d = load(case, "inspect-after-interruption.json")
    if not d:
        continue
    ws = d["work_state"]
    call = ws["calls"][0] if ws["calls"] else {}
    rows.append({
        "case": case,
        "stage": call.get("stage") or "-",
        "effect": call.get("provider_effect") or "-",
        "next": call.get("next_operation") or ws.get("task_next_operation") or "-",
        "dup_risk": call.get("duplicate_effect_risk"),
        "receipts": d["provider_receipts"],
        "ev_before": d["events_before"],
        "ev_after": d["events_after"],
    })

print(f"{'case':<38}{'stage':<20}{'effect':<10}{'next operation':<20}"
      f"{'dup?':<7}{'receipts':<10}{'events'}")
print("-" * 118)
for r in rows:
    ev = f"{r['ev_before']}->{r['ev_after']}"
    print(f"{r['case']:<38}{r['stage']:<20}{r['effect']:<10}{r['next']:<20}"
          f"{str(r['dup_risk']):<7}{r['receipts']:<10}{ev}")

case                                  stage               effect    next operation      dup?   receipts  events
----------------------------------------------------------------------------------------------------------------------
clean-complete                        completed           observed  none                False  1         14->14
compilation-failed                    -                   -         start_call          None   0         5->5
killed-after-manifest                 manifest_recorded   none      start_call          False  0         7->7
killed-after-manifest-drift           manifest_recorded   none      start_call          False  0         7->7
killed-after-observation              observed            observed  reinterpret         False  1         9->9
killed-during-compilation             -                   -         start_call          None   0         4->4
killed-during-effect                  effect_unknown      unknown   reconcile_effect    True   1         8-

## Observation

Two columns carry the chapter.

**`effect`** is the question that governs resume. It is not "did this step
finish?" — it is *could the effect have happened?* The projection derives it
from which events are present, using no clock and no provider query.

**`events`** shows `before -> after` for the inspecting process. They are
always equal: **reading the process's state changed nothing and asked the model
nothing.**

In [4]:
assert all(r["ev_before"] == r["ev_after"] for r in rows)
print("assertion held: every inspection appended 0 events")
print()
for r in rows:
    if r["effect"] == "unknown":
        print(f"the dangerous rung: {r['case']}")
        d = load(r["case"], "inspect-after-interruption.json")
        call = d["work_state"]["calls"][0]
        print("   stage  :", call["stage"])
        print("   reason :", call["reason"])
        print("   next   :", call["next_operation"])

assertion held: every inspection appended 0 events

the dangerous rung: killed-during-effect
   stage  : effect_unknown
   reason : attempt started with no observation: the provider may or may not have served the request; repeating could duplicate the effect
   next   : reconcile_effect
the dangerous rung: killed-during-effect-naive-restart
   stage  : effect_unknown
   reason : attempt started with no observation: the provider may or may not have served the request; repeating could duplicate the effect
   next   : reconcile_effect


That row is not a failure with a retry button. It is a question the ledger
leaves open **on purpose**, with the intent still readable beside it.

## 3. Resume once

The producer was killed right after `call.manifest`. No attempt had started, so
no provider effect was possible.

In [5]:
case = "killed-after-manifest"
res = load(case, "action-resume.json")

print(f"=== {case} ===")
print("  outcome        :", res.get("outcome"), "|", res.get("status"))
print("  receipts before:", res["receipts_before"])
print("  receipts after :", res["receipts_after"])
print("  events  before :", res["events_before"], "-> after:", res["events_after"])
print("  new call id    :", res.get("new_call_id"))

after = load(case, "inspect-after-resume.json")
call = after["work_state"]["calls"]
print()
print("  calls after resume:")
def short(x, n=10):
    return "-" if not x else (x if len(x) <= n else x[:n] + "...")

for c in call:
    print(f"    {short(c['call_id'],12):<16}stage={c['stage']:<18}"
          f"status={str(c['call_status']):<12}superseded_by={short(c.get('superseded_by'),12)}")

assert res["receipts_after"] == 1
print()
print("assertion held: exactly ONE provider request in total")

=== killed-after-manifest ===
  outcome        : resumed | succeeded
  receipts before: 0
  receipts after : 1
  events  before : 7 -> after: 17
  new call id    : 401e7bd8-b3b1-40a2-9980-4f56f20f8adc

  calls after resume:
    call-16         stage=manifest_recorded status=None        superseded_by=401e7bd8-b3b...
    401e7bd8-b3b... stage=completed         status=succeeded   superseded_by=-

assertion held: exactly ONE provider request in total


## 4. Refuse twice

### Refused for drift

The same interruption, resumed with a **different model** configured.

In [6]:
case = "killed-after-manifest-drift"
res = load(case, "action-resume.json")
print(f"=== {case} ===")
for k in ("outcome", "status", "model", "reason"):
    if k in res:
        print(f"  {k:<16}{res[k]}")
print("  receipts        :", res["receipts_before"], "->", res["receipts_after"])
print("  events          :", res["events_before"], "->", res["events_after"])
print()
print("Resuming does not mean running whatever the code would send today.")
print("It means running what was RECORDED as intended, or refusing.")

=== killed-after-manifest-drift ===
  outcome         refused
  model           a-different-model
  reason          recorded intent does not reproduce: request_body_sha256
  receipts        : 0 -> 0
  events          : 7 -> 7

Resuming does not mean running whatever the code would send today.
It means running what was RECORDED as intended, or refusing.


### Refused for an unknown effect

The producer was killed **while the provider held the request**. One receipt was
already in the provider's log; only `attempt.started` was in the ledger.

In [7]:
case = "killed-during-effect"
res = load(case, "action-resume.json")
print(f"=== {case} ===")
for k in ("outcome", "next_operation", "reason"):
    if k in res:
        print(f"  {k:<18}{res[k]}")
print("  receipts          :", res["receipts_before"], "->", res["receipts_after"])
print("  events            :", res["events_before"], "->", res["events_after"])

assert res["receipts_before"] == res["receipts_after"] == 1
assert res["events_before"] == res["events_after"]
print()
print("assertion held: resume refused, nothing appended, no second effect")

=== killed-during-effect ===
  outcome           refused
  next_operation    reconcile_effect
  reason            attempt started with no observation: the provider may or may not have served the request; repeating could duplicate the effect
  receipts          : 1 -> 1
  events            : 8 -> 8

assertion held: resume refused, nothing appended, no second effect


## 5. The naive restart

A copy of that same directory was handed to a process that did what restart
loops usually do: take the recorded request and issue it again **without
consulting the work state**.

In [8]:
case = "killed-during-effect-naive-restart"
res = load(case, "action-naive-restart.json")
print(f"=== {case} ===")
print("  idempotency key :", res.get("idempotency_key"))
print("  status          :", res.get("status"))
print("  receipts before :", res["receipts_before"])
print("  receipts after  :", res["receipts_after"])
print("  note            :", (res.get("note") or "")[:110])

assert res["receipts_after"] == 2
print()
print("assertion held: TWO provider requests, from two processes,")
print("under the same idempotency key.")

=== killed-during-effect-naive-restart ===
  idempotency key : key-16
  status          : succeeded
  receipts before : 1
  receipts after  : 2
  note            : re-issued the recorded request under the same idempotency key without consulting work state

assertion held: TWO provider requests, from two processes,
under the same idempotency key.


## Observation

Put the two side by side. Same interruption, same ledger, same key.

In [9]:
resume = load("killed-during-effect", "action-resume.json")
naive  = load("killed-during-effect-naive-restart", "action-naive-restart.json")

print(f"{'strategy':<28}{'consulted work state':<24}{'receipts before':<18}{'after'}")
print("-" * 82)
print(f"{'resume (refused)':<28}{'yes':<24}{resume['receipts_before']:<18}"
      f"{resume['receipts_after']}")
print(f"{'naive restart':<28}{'no':<24}{naive['receipts_before']:<18}"
      f"{naive['receipts_after']}")
print()
print("The idempotency key did not help, because replay only matches calls")
print("that COMPLETED, and this one never had.")
print()
print("The ledger does record both calls, so the duplicate is visible")
print("afterwards. Nothing prevented it.")

strategy                    consulted work state    receipts before   after
----------------------------------------------------------------------------------
resume (refused)            yes                     1                 1
naive restart               no                      1                 2

The idempotency key did not help, because replay only matches calls
that COMPLETED, and this one never had.

The ledger does record both calls, so the duplicate is visible
afterwards. Nothing prevented it.


## 6. Observed, but not automated

The producer was killed after the observation was committed. The response bytes
are preserved, so the provider is not needed again.

In [10]:
case = "killed-after-observation"
d = load(case, "inspect-after-interruption.json")
call = d["work_state"]["calls"][0]
print(f"  stage          : {call['stage']}")
print(f"  next operation : {call['next_operation']}")
print(f"  provider effect: {call['provider_effect']}")
print(f"  reason         : {call['reason'][:96]}")

res = load(case, "action-resume.json")
if res:
    print()
    print("  resume outcome :", res.get("outcome"))
    print("  receipts       :", res["receipts_before"], "->", res["receipts_after"])
print()
print("This one is not unsafe. CodeAI simply does not yet automate re-running")
print("the interpreter over stored bytes. The distinction is RECORDED, not")
print("blurred - and Chapter 17 is where it gets closed.")

  stage          : observed
  next operation : reinterpret
  provider effect: observed
  reason         : transport observation preserved; interpretation missing

  resume outcome : refused
  receipts       : 1 -> 1

This one is not unsafe. CodeAI simply does not yet automate re-running
the interpreter over stored bytes. The distinction is RECORDED, not
blurred - and Chapter 17 is where it gets closed.


## 7. Compilations that stopped or failed

In [11]:
for case in ("killed-during-compilation", "compilation-failed"):
    d = load(case, "inspect-after-interruption.json")
    if not d:
        continue
    comps = d["work_state"]["compilations"]
    print(f"=== {case} ===")
    for c in comps:
        print(f"  next operation    : {c['next_operation']}")
        print(f"  offered ids       : {len(c.get('offered_ids') or c.get('included_ids') or [])}")
        print(f"  missing required  : {c.get('missing_required_ids')}")
        print(f"  failure           : {str(c.get('failure'))[:80]}")
    print()

print("Chapter 15 found exactly this information missing from the runtime.")
print("Recompiling is always safe, because compiling has no external effect.")

=== killed-during-compilation ===
  next operation    : recompile
  offered ids       : 3
  missing required  : []
  failure           : None

=== compilation-failed ===
  next operation    : fix_inputs
  offered ids       : 3
  missing required  : ['never-offered']
  failure           : RequiredContextMissing

Chapter 15 found exactly this information missing from the runtime.
Recompiling is always safe, because compiling has no external effect.


## Interpretation

The chapter's rule, now with receipts behind it:

> Mark the point just before each external effect. Between that mark and the
> observation, the honest answer is **unknown**.

Three consequences:

1. **Resume only where no effect was possible**, and resume **what was
   intended** — rebuild the recorded request and refuse if it no longer
   matches.
2. **Where an effect may have happened, refuse and reconcile.** Never retry on
   a guess. An idempotency key does not catch a call that never completed.
3. **Process memory is not model memory.** A model that remembers more can
   still be run twice by a process that does not know what it was doing.

And the limit the chapter keeps visible: "reconcile the effect" *names* the
problem without a mechanism. CodeAI has no way to ask the provider whether a
request was served. The refusal is the whole recovery plan.

## Try it yourself

1. **Classify a case yourself.** Open `events-after-interruption.json` for
   `killed-during-effect` and derive the stage from which event kinds are
   present. You are re-implementing the projection's ladder.
2. **Find the orphan.** Which event appears with no partner? That asymmetry is
   the entire signal.
3. **Break the order.** Imagine `attempt.started` were appended *after* the
   request instead of before. Re-derive every row in section 2. How many cases
   become indistinguishable?
4. **Price the duplicate.** The naive restart cost one extra provider request
   here. Multiply by your own retry loop's rate and your own per-call price.